# RF-DETR — inference → para GT–pred → EigenCAM / GradCAM++

**Wymaga:** `uv sync --extra rfdetr-xai`, checkpoint RF-DETR oraz JSON COCO + obrazy.

Poniżej szablon komórek: uzupełnij ścieżki i uruchom na jednym obrazie.

In [ ]:
from pathlib import Path
import sys

SRC = Path("../src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from PIL import Image
from rfdetr_load import load_rfdetr
from rfdetr_eval_config import greedy_match_single_class, coco_bbox_xywh_to_xyxy
from rfdetr_gradcam import run_cam_and_save

# --- UZUPEŁNIJ ---
CHECKPOINT = Path("/ścieżka/do/best_model.pt")
MODEL_SIZE = "large"
IMAGE_PATH = Path("/ścieżka/do/obrazu.jpg")
GT_XYXY = (100.0, 80.0, 220.0, 190.0)  # lub z COCO po wczytaniu adnotacji
CLASS_ID_MODEL = 0

model = load_rfdetr(MODEL_SIZE, CHECKPOINT)
pil = Image.open(IMAGE_PATH).convert("RGB")
det = model.predict(pil, threshold=0.25)
print("Detekcje:", len(det) if det is not None else 0)

out_dir = Path("../wyniki/rfdetr_xai/notebook_demo")
run_cam_and_save(
    model,
    pil,
    matched_box_xyxy_orig=GT_XYXY,
    class_id_model=CLASS_ID_MODEL,
    out_png=out_dir / "eigen_cam.png",
    method="eigen",
    metadata_extra={"image": str(IMAGE_PATH)},
)
run_cam_and_save(
    model,
    pil,
    matched_box_xyxy_orig=GT_XYXY,
    class_id_model=CLASS_ID_MODEL,
    out_png=out_dir / "gradcamplusplus.png",
    method="gradcampp",
    metadata_extra={"image": str(IMAGE_PATH)},
)
print("Zapisano mapy w", out_dir)